In [1]:
from ingest import load_faq_data
documents = load_faq_data()

In [2]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

103

In [3]:
documents = documents_llm

In [4]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [5]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [6]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [7]:
from openai import OpenAI
import os

openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)
model="openai/gpt-oss-120b"

In [8]:
import json

user_prompt = json.dumps(doc)

In [9]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [10]:
response = openai_client.responses.parse(
    model=model,
    input=messages,
    text_format=Questions
)

In [11]:
result = response.output_parsed

print(result)
result.questions

questions=['Can I enroll in the LLM ZoomCamp now that it’s already started?', 'Is it still possible to get a certificate if I join late?', 'What’s the deadline for submitting the final project to earn the certificate?', 'Do I need to finish all modules before submitting the project for certification?', 'If I sign up now, can I still access all the course materials and assignments?']


['Can I enroll in the LLM ZoomCamp now that it’s already started?',
 'Is it still possible to get a certificate if I join late?',
 'What’s the deadline for submitting the final project to earn the certificate?',
 'Do I need to finish all modules before submitting the project for certification?',
 'If I sign up now, can I still access all the course materials and assignments?']

In [12]:
from evaluation_utils import llm_structured

/workspaces/LLM-zoomcamp-04-evaluation/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions,
    model=model
)

result.questions
usage.input_tokens, usage.output_tokens

(336, 493)

In [16]:
usage

ResponseUsage(input_tokens=336, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=493, output_tokens_details=OutputTokensDetails(reasoning_tokens=406), total_tokens=829)

In [22]:
from evaluation_utils import calc_total_price_gpt_5_4_mini

In [23]:
total_cost = calc_total_price_gpt_5_4_mini([usage])

In [24]:
total_cost

0.0024705

In [25]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'Can I still join the LLM Zoomcamp if I just found out about it?',
  'document': '74eb249bbf'},
 {'question': 'Is it possible to enroll after the course has started?',
  'document': '74eb249bbf'},
 {'question': 'Do I need to enroll early to get a certificate?',
  'document': '74eb249bbf'},
 {'question': 'Can I still submit my project for a certificate even if I join late?',
  'document': '74eb249bbf'},
 {'question': 'What are the deadlines for project submission to receive a certificate?',
  'document': '74eb249bbf'}]

In [26]:
import pandas as pd

In [27]:
pd.DataFrame(records)

,question,document
0,Can I still join the LLM Zoomcamp if I just fo...,74eb249bbf
1,Is it possible to enroll after the course has ...,74eb249bbf
2,Do I need to enroll early to get a certificate?,74eb249bbf
3,Can I still submit my project for a certificat...,74eb249bbf
4,What are the deadlines for project submission ...,74eb249bbf


In [28]:
from evaluation_utils import llm_structured_retry

In [29]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [ ]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)